# 03 — Two-stage anchor-based: Faster R-CNN

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q faster-coco-eval

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image

from vod import CLASSES
from vod.data import find_data_root, load_split
from vod.errors import match_detections, outcome_tables
from vod.protocol import PROTOCOL
from vod.submission import write_submission
from vod.tv import (FrameDataset, comparison_tables, completed_runs, load_run_model, predict_test,
                    proposal_sweep, run_all, two_stage_breakdown)

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")

## Data (same protocol as every detector)

In [ ]:
train_frames, train_targets = load_split(DATA / "train", step=PROTOCOL["train_step"])
val_frames, val_targets = load_split(DATA / "val", step=1)
VAL_GT = DATA / "val" / "annotations_coco.json"
print(len(train_frames), "train frames,", len(val_frames), "val frames")

## Experiments
torchvision `fasterrcnn_resnet50_fpn_v2`, COCO-pretrained, SGD lr 0.005 (0.02 × 4/16, torchvision
reference scaled to batch 4), warmup + cosine, AMP. Each experiment changes one thing vs the default.
The number of RPN proposals is studied at test time on the trained default model (no retraining).

In [ ]:
BASE = dict(arch="faster_rcnn", lr=0.005)
EXPERIMENTS = {
    "frcnn_default": BASE,
    # RoI sampling: 50% positives instead of 25% — dense frames have enough foreground proposals
    "frcnn_roi_pos50": {**BASE, "roi_pos_fraction": 0.5},
    # RPN anchors one octave smaller (16-256 instead of 32-512), as for RetinaNet
    "frcnn_anchors": {**BASE, "anchor_sizes": (16, 32, 64, 128, 256)},
    # loss modification: double weight of the second-stage box regression loss (AP75 is the weak point)
    "frcnn_boxloss2": {**BASE, "loss_weights": {"loss_box_reg": 2.0}},
}
RUN = list(EXPERIMENTS)  # ~2 h each on a T4; finished runs are saved right away

## Train and evaluate (last epoch, every val frame)

In [ ]:
run_all(RUN, EXPERIMENTS, PROTOCOL, train_frames, train_targets, val_frames, val_targets, VAL_GT, WORK)

## Comparison

In [ ]:
runs = completed_runs((WORK, Path("/kaggle/input")))
main, per_class, per_clip = comparison_tables(runs, CLASSES)
display(main)
display(per_class)
per_clip  # mAP of each val clip: is a difference consistent across scenes?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for n, m in runs.items():
    if m.get("history") is not None:
        ax.plot(m["history"]["epoch"], m["history"]["mAP"], marker="o", label=n)
ax.set(xlabel="epoch", ylabel="val mAP (every 10th frame)", title="Validation mAP during training")
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(WORK / "map_curves.png", dpi=110)

## Analysis of the best Faster R-CNN

In [ ]:
frcnn = main[main.index.isin(EXPERIMENTS)]
BEST = frcnn["mAP"].idxmax()
cfg = {**PROTOCOL, **EXPERIMENTS[BEST]}
model = load_run_model(BEST, cfg, (WORK, Path("/kaggle/input")))
print("best:", BEST, "| val mAP", frcnn.loc[BEST, "mAP"])

### What each stage contributes
RPN columns: share of objects covered (IoU >= 0.5) by the top-n proposals and the median best
proposal IoU. Final columns: the same for the second stage's detections (score >= 0.05).

In [ ]:
val_ds = FrameDataset(val_frames, val_targets, cfg)
two_stage_breakdown(model, val_ds)

### Number of RPN proposals at test time (same weights, every 5th val frame)

In [ ]:
proposal_sweep(model, FrameDataset(val_frames.iloc[::5], val_targets, cfg), VAL_GT, cfg)

### Failure analysis
Every object and every detection with score >= 0.3, matched at IoU 0.5.

In [ ]:
val_dets = pd.read_parquet(runs[BEST]["dir"] / "val_dets.parquet")
objects, detections = match_detections(DATA / "val", val_dets, score_thr=0.3)
by_size, by_class, det_share = outcome_tables(objects, detections, CLASSES)
display(by_size)
display(by_class)
det_share

In [ ]:
def show_failures(key, ax, title):
    """Missed objects red, wrong class orange, poor box yellow; background false positives blue."""
    o, d = objects[objects["key"] == key], detections[detections["key"] == key]
    ax.imshow(Image.open(val_frames.loc[val_frames["file_name"].str.contains(key, regex=False), "path"].iloc[0]))
    colors = {"missed": "red", "wrong_class": "orange", "poor_box": "yellow", "detected": "lime"}
    for r in o.itertuples():
        ax.add_patch(Rectangle((r.x, r.y), r.w, r.h, fill=False, ec=colors[r.outcome], lw=0.8))
    for r in d[d["outcome"] == "background"].itertuples():
        ax.add_patch(Rectangle((r.x, r.y), r.w, r.h, fill=False, ec="deepskyblue", lw=0.8))
    ax.set_title(title, fontsize=9)
    ax.axis("off")


per_frame = objects.groupby("key")["outcome"].value_counts().unstack(fill_value=0)
fp_frame = detections[detections["outcome"] == "background"].groupby("key").size()
cases = {
    "most missed objects": per_frame["missed"].idxmax(),
    "most wrong-class objects": per_frame["wrong_class"].idxmax() if "wrong_class" in per_frame else None,
    "most background false positives": fp_frame.idxmax() if len(fp_frame) else None,
}
fig, axes = plt.subplots(len(cases), 1, figsize=(16, 9 * len(cases)))
for ax, (title, key) in zip(axes, cases.items()):
    if key is not None:
        show_failures(key, ax, f"{title}: {key}")
plt.tight_layout()
plt.savefig(WORK / "failure_cases.png", dpi=100)

## Test predictions with the best Faster R-CNN

In [ ]:
sub = write_submission(predict_test(model, cfg, DATA), DATA / "test_images.csv", WORK / f"submission_{BEST}.csv")
sub.head()